# Découpage train / validation / test

**À exécuter une seule fois.** Ce notebook produit les trois partitions utilisées par toutes les expérimentations de l'étape 2. Le notebook d'entraînement les lit sans jamais les régénérer.

| Partition | Part | Rôle |
|---|---|---|
| `train.parquet` | 70 % | Apprentissage et ajustement des prétraitements |
| `val.parquet` | 15 % | Comparaison des modèles, optimisation des hyperparamètres |
| `test.parquet` | 15 % | Évaluation finale du seul modèle retenu |

`test` est relisible à volonté mais n'intervient dans **aucune décision** de modélisation, sous peine de perdre son indépendance. Si la préparation des données change, les partitions sont à régénérer et l'entraînement à relancer : les empreintes `data_version` et `split_version` calculées ici rendent cette dépendance visible dans chaque run MLflow.

## 1. Chargement et empreinte des données

In [1]:
# Le projet est installe en mode editable par `uv sync`. Repli au cas ou le
# noyau selectionne ne serait pas celui du projet : on ajoute src/ au chemin.
try:
    import agritech  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    _racine = next(p for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / 'pyproject.toml').exists())
    sys.path.insert(0, str(_racine / 'src'))

import hashlib
import json
from datetime import datetime, timezone

import pandas as pd
from sklearn.model_selection import train_test_split

from agritech.config import (
    CONSOLIDATED_PARQUET_PATH,
    SPLITS_DIR,
    SPLIT_METADATA_PATH,
    SPLIT_PROPORTIONS,
    SPLIT_RANDOM_STATE,
    SPLIT_STRATIFY_ON,
    TARGET,
    TEST_PATH,
    TRAIN_PATH,
    VAL_PATH,
)

CHEMINS = {'train': TRAIN_PATH, 'val': VAL_PATH, 'test': TEST_PATH}

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)


In [2]:
def empreinte(chemin, longueur=12):
    # SHA-256 par blocs : le fichier consolide fait plusieurs dizaines de Mo
    digest = hashlib.sha256()
    with chemin.open('rb') as fichier:
        for bloc in iter(lambda: fichier.read(1 << 20), b''):
            digest.update(bloc)
    return digest.hexdigest()[:longueur]


dataset = pd.read_parquet(CONSOLIDATED_PARQUET_PATH)
data_version = empreinte(CONSOLIDATED_PARQUET_PATH)

# Les controles de chevauchement raisonnent sur l'index : il doit etre unique
assert dataset.index.is_unique, 'Index non unique : les controles seraient fausses'

print('Dataset consolide :', f'{len(dataset):,}', 'lignes x', dataset.shape[1], 'colonnes')
print('data_version      :', data_version)
print()
print('Colonnes :', list(dataset.columns))

Dataset consolide : 999,769 lignes x 14 colonnes
data_version      : c9547cbd218b

Colonnes : ['Region', 'Soil_Type', 'Crop', 'Rainfall_mm', 'Temperature_Celsius', 'Fertilizer_Used', 'Irrigation_Used', 'Weather_Condition', 'Yield_tons_per_hectare', 'crop_productivity_index', 'crop_yield_geo_cv', 'crop_yield_trend_decade', 'crop_in_fao_reference', 'low_rain_no_irrigation']


## 2. Découpage stratifié

Le test est isolé d'abord, puis le reste séparé en entraînement et validation. La stratification par culture ne corrige aucun déséquilibre — `Crop` est uniformément répartie — mais garantit la présence des six cultures dans les trois jeux.

In [3]:
part_test = SPLIT_PROPORTIONS['test']
part_val_dans_reste = SPLIT_PROPORTIONS['val'] / (1 - part_test)

reste, test = train_test_split(
    dataset, test_size=part_test, random_state=SPLIT_RANDOM_STATE,
    stratify=dataset[SPLIT_STRATIFY_ON], shuffle=True)

train, val = train_test_split(
    reste, test_size=part_val_dans_reste, random_state=SPLIT_RANDOM_STATE,
    stratify=reste[SPLIT_STRATIFY_ON], shuffle=True)

partitions = {'train': train, 'val': val, 'test': test}

resume = pd.DataFrame({
    'lignes': {nom: len(d) for nom, d in partitions.items()},
    'part_%': {nom: round(100 * len(d) / len(dataset), 2) for nom, d in partitions.items()},
    'part_visee_%': {nom: 100 * p for nom, p in SPLIT_PROPORTIONS.items()},
})
resume.loc['total'] = [len(dataset), 100.0, 100.0]
resume

,lignes,part_%,part_visee_%
train,699837.0,70.0,70.0
val,149966.0,15.0,15.0
test,149966.0,15.0,15.0
total,999769.0,100.0,100.0


## 3. Contrôles d'intégrité

In [4]:
# Aucune observation ne doit appartenir a deux partitions
index_train, index_val, index_test = (set(partitions[n].index) for n in ('train', 'val', 'test'))

controles = {
    'total conserve': sum(len(d) for d in partitions.values()) == len(dataset),
    'train ∩ val vide': not (index_train & index_val),
    'train ∩ test vide': not (index_train & index_test),
    'val ∩ test vide': not (index_val & index_test),
    'union complete': len(index_train | index_val | index_test) == len(dataset),
}
for libelle, resultat in controles.items():
    print(('OK   ' if resultat else 'ECHEC') + ' | ' + libelle)

assert all(controles.values()), 'Un controle d integrite a echoue'

OK    | total conserve
OK    | train ∩ val vide
OK    | train ∩ test vide
OK    | val ∩ test vide
OK    | union complete


### 3.1 Comparabilité des distributions

Un équilibre sur une variable ne dit rien des autres, ni la moyenne sur la dispersion : modalités, moyennes, écarts-types, queues de la cible et valeurs manquantes sont contrôlés séparément.

In [5]:
# Variables qualitatives : part de chaque modalite dans chaque partition.
# La liste est DEDUITE des types, pour qu'un changement de colonnes dans la
# preparation ne laisse pas ce controle porter sur des variables disparues.
qualitatives = dataset.select_dtypes(include=['object', 'str', 'bool']).columns.tolist()
print('Qualitatives controlees :', qualitatives)

ecarts = []
for colonne in qualitatives:
    reference = dataset[colonne].value_counts(normalize=True)
    for nom, partition in partitions.items():
        part = partition[colonne].value_counts(normalize=True).reindex(reference.index).fillna(0)
        ecarts.append({'variable': colonne, 'partition': nom,
                       'ecart_max_pp': round(100 * (part - reference).abs().max(), 3)})

ecarts = pd.DataFrame(ecarts).pivot(index='variable', columns='partition', values='ecart_max_pp')
print("Ecart maximal a la distribution d'ensemble, en points de pourcentage :")
ecarts

Qualitatives controlees : ['Region', 'Soil_Type', 'Crop', 'Fertilizer_Used', 'Irrigation_Used', 'Weather_Condition', 'crop_in_fao_reference', 'low_rain_no_irrigation']
Ecart maximal a la distribution d'ensemble, en points de pourcentage :


partition,test,train,val
variable,,,
Crop,0.000,0.000,0.000
Fertilizer_Used,0.071,0.009,0.029
Irrigation_Used,0.136,0.058,0.136
Region,0.235,0.047,0.132
Soil_Type,0.101,0.024,0.199
Weather_Condition,0.250,0.066,0.056
crop_in_fao_reference,0.000,0.000,0.000
low_rain_no_irrigation,0.001,0.023,0.110


In [6]:
# Variables quantitatives : position (moyenne) ET dispersion (ecart-type).
# Deux partitions peuvent partager la meme moyenne et differer par la dispersion.
# Liste deduite des types, cible comprise.
quantitatives = dataset.select_dtypes(include='number').columns.tolist()

lignes = []
for colonne in quantitatives:
    moy_ref, std_ref = dataset[colonne].mean(), dataset[colonne].std()
    moyennes = [partition[colonne].mean() for partition in partitions.values()]
    dispersions = [partition[colonne].std() for partition in partitions.values()]
    lignes.append({
        'variable': colonne,
        'moyenne': round(moy_ref, 4),
        # ecarts mesures avant arrondi, sinon on mesurerait l'arrondi lui-meme
        'ecart_max_moyenne': round(max(abs(m - moy_ref) for m in moyennes), 5),
        'ecart_type': round(std_ref, 4),
        'ecart_max_ecart_type': round(max(abs(s - std_ref) for s in dispersions), 5),
    })

print("Ecart maximal entre une partition et l'ensemble :")
pd.DataFrame(lignes).set_index('variable')

Ecart maximal entre une partition et l'ensemble :


,moyenne,ecart_max_moyenne,ecart_type,ecart_max_ecart_type
variable,,,,
Rainfall_mm,550.0778,0.53244,259.8043,0.57400
Temperature_Celsius,27.5061,0.01973,7.2205,0.01496
Yield_tons_per_hectare,4.6506,0.00135,1.6952,0.00424
crop_productivity_index,0.9977,0.00000,0.2839,0.00000
crop_yield_geo_cv,0.5312,0.00000,0.0878,0.00000
crop_yield_trend_decade,0.3172,0.00000,0.1022,0.00000


In [7]:
# Queues de la cible : un desequilibre des valeurs extremes ne se verrait
# ni dans la moyenne ni dans l'ecart-type
niveaux = [0.01, 0.25, 0.50, 0.75, 0.99]
quantiles = pd.DataFrame({nom: partition[TARGET].quantile(niveaux)
                          for nom, partition in partitions.items()})
quantiles['ensemble'] = dataset[TARGET].quantile(niveaux)

print('Quantiles de', TARGET, ':')
quantiles.round(4)

Quantiles de Yield_tons_per_hectare :


,train,val,test,ensemble
0.01,0.9971,0.9861,1.0085,0.9971
0.25,3.4191,3.4102,3.4249,3.4187
0.50,4.6532,4.6514,4.6487,4.6523
0.75,5.8802,5.8851,5.8710,5.8796
0.99,8.3121,8.3250,8.2997,8.3124


In [8]:
# Valeurs manquantes : la proportion doit etre stable d'une partition a l'autre
manquants = pd.DataFrame({
    nom: (100 * partition.isna().mean()).round(2)
    for nom, partition in partitions.items()
})
manquants['ensemble'] = (100 * dataset.isna().mean()).round(2)
manquants[manquants.sum(axis=1) > 0]

,train,val,test,ensemble
crop_productivity_index,33.34,33.34,33.33,33.34
crop_yield_geo_cv,33.34,33.34,33.33,33.34
crop_yield_trend_decade,33.34,33.34,33.33,33.34


> **Lecture.** On attend des écarts de l'ordre du hasard d'échantillonnage, pas des écarts nuls :
> sur 150 000 lignes, une proportion de 15 % fluctue déjà de ±0,09 point. Quelques dixièmes de
> point sont normaux, plusieurs points signaleraient une erreur.
>
> Le dernier tableau prime : une proportion de valeurs manquantes instable signifierait que les
> cultures hors référentiel FAO sont inégalement réparties, et les deux jeux de variables de
> l'étape 2 ne seraient plus comparables.

## 4. Écriture des partitions

Quatre cas, pour qu'un « Run All » ne puisse jamais invalider les runs MLflow en cours :

- **rien sur disque** → écriture ;
- **lot complet et à jour** (trois partitions *et* métadonnées, même `data_version`) → écriture ignorée, fichiers existants vérifiés ;
- **lot périmé**, généré pour un autre dataset → arrêt avec le diagnostic ; c'est le cas après toute modification de la préparation ;
- **lot incomplet**, exécution interrompue → arrêt, les fichiers pouvant venir de découpages différents.

Une régénération volontaire demande de passer `AUTORISER_ECRASEMENT` à `True`.

In [ ]:
# Verrou : a False, un « Run All » ne peut pas reecrire les partitions et
# invalider les runs MLflow qui s'y referent. Ne passer a True que pour une
# regeneration voulue, puis le remettre a False.
AUTORISER_ECRASEMENT = False

SPLITS_DIR.mkdir(parents=True, exist_ok=True)

# Les metadonnees font partie du lot : des partitions sans leur split_version
# ne sont pas exploitables par MLflow
ARTEFACTS = {**CHEMINS, 'metadata': SPLIT_METADATA_PATH}
presents = [nom for nom, chemin in ARTEFACTS.items() if chemin.exists()]

# Les partitions presentes decrivent-elles le dataset ACTUEL ? Leur simple
# existence ne suffit pas : si la preparation des donnees a change, elles sont
# perimees et doivent etre regenerees.
precedent = (json.loads(SPLIT_METADATA_PATH.read_text(encoding='utf-8'))
             if 'metadata' in presents else None)
perimees = precedent is not None and precedent['data_version'] != data_version

if not presents:
    ecrire = True
    print('Aucun artefact existant : ecriture.')

elif AUTORISER_ECRASEMENT:
    ecrire = True
    print('ATTENTION - ecrasement explicitement autorise :', presents)

elif len(presents) < len(ARTEFACTS):
    # Vestige d'une execution interrompue : les fichiers presents peuvent
    # provenir de decoupages differents. On refuse de deviner.
    raise RuntimeError(
        'Etat partiel. Presents : ' + ', '.join(presents) + ' | manquants : '
        + ', '.join(n for n in ARTEFACTS if n not in presents)
        + '. Supprimer le dossier splits/ et relancer, ou passer '
        'AUTORISER_ECRASEMENT a True.')

elif perimees:
    # Le cas le plus courant apres une modification de la preparation.
    # On le signale ICI, avec le diagnostic, plutot que de laisser echouer
    # la verification finale trois cellules plus loin.
    raise RuntimeError(
        'PARTITIONS PERIMEES — elles ont ete generees pour un autre dataset.\n'
        f"  sur disque     : data_version {precedent['data_version']}, "
        f"{len(precedent['colonnes'])} colonnes, genere le {precedent['genere_le']}\n"
        f'  dataset actuel : data_version {data_version}, '
        f'{dataset.shape[1]} colonnes\n'
        '  La preparation des donnees a change depuis. Passer\n'
        '  AUTORISER_ECRASEMENT a True en haut de cette cellule, puis relancer.')

else:
    ecrire = False
    print('Lot complet et a jour : ecriture ignoree, verification seule.')

In [10]:
if ecrire:
    for nom, partition in partitions.items():
        partition.to_parquet(CHEMINS[nom], index=False)

for nom, chemin in CHEMINS.items():
    print(f'{chemin.name:<16} {chemin.stat().st_size / 1024 ** 2:>7.1f} Mo')

# Empreinte des octets des trois fichiers, dans un ordre fixe. Elle changerait si
# pyarrow les reecrivait differemment, meme a donnees identiques : c'est voulu,
# puisqu'elle identifie les artefacts reellement lus par l'entrainement.
combinee = ''.join(empreinte(c, 64) for c in CHEMINS.values())
split_version = hashlib.sha256(combinee.encode()).hexdigest()[:12]

print()
print('data_version  :', data_version)
print('split_version :', split_version)

train.parquet       18.6 Mo
val.parquet          4.6 Mo
test.parquet         4.6 Mo

data_version  : c9547cbd218b
split_version : ccb2a631078f


In [11]:
metadonnees = {
    'data_version': data_version,
    'split_version': split_version,
    'source': CONSOLIDATED_PARQUET_PATH.name,
    'random_state': SPLIT_RANDOM_STATE,
    'stratify_on': SPLIT_STRATIFY_ON,
    'proportions': SPLIT_PROPORTIONS,
    'effectifs': {nom: len(d) for nom, d in partitions.items()},
    'colonnes': list(dataset.columns),
    'genere_le': datetime.now(timezone.utc).isoformat(timespec='seconds'),
}

if ecrire:
    SPLIT_METADATA_PATH.write_text(json.dumps(metadonnees, indent=2), encoding='utf-8')
    print('Metadonnees ecrites dans', SPLIT_METADATA_PATH.name)
else:
    # Rien n'a ete reecrit : les metadonnees du disque font foi, et la section 5
    # verifiera qu'elles decrivent bien les fichiers presents
    metadonnees = json.loads(SPLIT_METADATA_PATH.read_text(encoding='utf-8'))
    print('Metadonnees relues depuis', SPLIT_METADATA_PATH.name)

print()
print(json.dumps(metadonnees, indent=2))

Metadonnees ecrites dans split_metadata.json

{
  "data_version": "c9547cbd218b",
  "split_version": "ccb2a631078f",
  "source": "crop_yield_consolidated.parquet",
  "random_state": 42,
  "stratify_on": "Crop",
  "proportions": {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15
  },
  "effectifs": {
    "train": 699837,
    "val": 149966,
    "test": 149966
  },
  "colonnes": [
    "Region",
    "Soil_Type",
    "Crop",
    "Rainfall_mm",
    "Temperature_Celsius",
    "Fertilizer_Used",
    "Irrigation_Used",
    "Weather_Condition",
    "Yield_tons_per_hectare",
    "crop_productivity_index",
    "crop_yield_geo_cv",
    "crop_yield_trend_decade",
    "crop_in_fao_reference",
    "low_rain_no_irrigation"
  ],
  "genere_le": "2026-10-25T10:40:26+00:00"
}


## 5. Relecture et vérification finale

In [12]:
# Les fichiers sur disque doivent correspondre exactement aux partitions calculees ici
relues = {nom: pd.read_parquet(chemin) for nom, chemin in CHEMINS.items()}

conformes = {}
for nom, relue in relues.items():
    attendue = partitions[nom]
    conformes[nom] = (
        len(relue) == len(attendue)
        and list(relue.columns) == list(dataset.columns)
        and relue[TARGET].notna().all()
        and float((relue[TARGET] - attendue[TARGET].reset_index(drop=True)).abs().max()) < 1e-12)
    print(('OK   ' if conformes[nom] else 'ECHEC') + f' | {nom:<6} {len(relue):>9,} lignes, '
          f'{relue.shape[1]} colonnes, cible complete et identique')

# Les metadonnees doivent decrire les fichiers reellement presents
coherent = (metadonnees['split_version'] == split_version
            and metadonnees['effectifs'] == {nom: len(r) for nom, r in relues.items()}
            and metadonnees['colonnes'] == list(dataset.columns))
print(('OK   ' if coherent else 'ECHEC') + ' | metadonnees coherentes avec les fichiers')

# Bloquant : ces partitions servent de reference a toutes les experiences MLflow
assert all(conformes.values()) and coherent, 'La verification des partitions a echoue'

OK    | train    699,837 lignes, 14 colonnes, cible complete et identique
OK    | val      149,966 lignes, 14 colonnes, cible complete et identique
OK    | test     149,966 lignes, 14 colonnes, cible complete et identique
OK    | metadonnees coherentes avec les fichiers


## À retenir

Partitions écrites et relues sans perte, contrôles finaux bloquants. **Ce notebook n'a plus à être exécuté**, sauf changement du dataset consolidé — qui imposerait de relancer entraînement et évaluation, et de traiter les runs antérieurs comme une autre version expérimentale.

`data_version` et `split_version` sont à reporter sur chaque run MLflow : deux runs d'empreintes différentes ne sont pas comparables.